In [12]:
!pip install --upgrade openai openai-agents python-dotenv
!pip install --upgrade agents mcp
!npm update

   ---------------------------------------- 0.0/807.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/807.4 kB ? eta -:--:--
   ---------------------------------------- 807.4/807.4 kB 5.9 MB/s  0:00:00
  Attempting uninstall: openai-agents
    Found existing installation: openai-agents 0.14.1
    Uninstalling openai-agents-0.14.1:
      Successfully uninstalled openai-agents-0.14.1


  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
     ---------------------------------------- 0.0/721.7 kB ? eta -:--:--
     ---------------------------------------- 721.7/721.7 kB 7.3 MB/s  0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for agents: filename=agents-1.4.0-py3-none-any.whl size=62751 sha256=ce27ab609ae5fdce48f170b19e71d1e2fdeb0cb2e9f6e97a336e61761f310257
  Stored in directory: c:\users\panka\

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-core 0.1.53 requires packaging<24.0,>=23.2, but you have packaging 26.1 which is incompatible.
langchain-mcp 0.2.1 requires langchain-core~=0.3.37, but you have langchain-core 0.1.53 which is incompatible.
langchain-openai 0.0.6 requires numpy<2,>=1, but you have numpy 2.4.4 which is incompatible.
langchain-openai 0.0.6 requires openai<2.0.0,>=1.10.0, but you have openai 2.32.0 which is incompatible.
langgraph-checkpoint 4.0.1 requires langchain-core>=0.2.38, but you have langchain-core 0.1.53 which is incompatible.
langgraph-prebuilt 1.0.8 requires langchain-core>=1.0.0, but you have langchain-core 0.1.53 which is incompatible.



up to date in 1s


In [2]:
server_code = """
import asyncio

from agents import Agent, Runner
from agents.mcp import MCPServerStdio


async def main() -> None:
    async with MCPServerStdio(
        name=\"Codex CLI\",
        params={
            \"command\": \"npx\",
            \"args\": [\"-y\", \"codex\", \"mcp-server\"],
        },
        client_session_timeout_seconds=360000,
    ) as codex_mcp_server:
        print(\"Codex MCP server started.\")
        # More logic coming in the next sections.
        return


if __name__ == "__main__":
    asyncio.run(main())
"""
with open("codex_mcp.py", "w") as f:
    f.write(server_code)

print("codex_mcp.py written")

codex_mcp.py written


In [13]:
!python codex_mcp.py

node:internal/modules/esm/get_format:219
  throw new ERR_UNKNOWN_FILE_EXTENSION(ext, filepath);
        ^

TypeError [ERR_UNKNOWN_FILE_EXTENSION]: Unknown file extension ".py" for D:\Ethans\Python\AI-Automation\codex_mcp.py
    at Object.getFileProtocolModuleFormat [as file:] (node:internal/modules/esm/get_format:219:9)
    at defaultGetFormat (node:internal/modules/esm/get_format:245:36)
    at defaultLoad (node:internal/modules/esm/load:98:16)
    at ModuleLoader.load (node:internal/modules/esm/loader:815:12)
    at ModuleLoader.loadAndTranslate (node:internal/modules/esm/loader:594:31)
    at #createModuleJob (node:internal/modules/esm/loader:624:36)
    at #getJobFromResolveResult (node:internal/modules/esm/loader:343:34)
    at ModuleLoader.getModuleJobForImport (node:internal/modules/esm/loader:311:41)
    at async onImport.tracePromise.__proto__ (node:internal/modules/esm/loader:664:25) {
  code: 'ERR_UNKNOWN_FILE_EXTENSION'
}

Node.js v22.22.0


Build a single-agent workflow

Let’s start with a scoped example that uses Codex MCP to ship a small browser game. The workflow relies on two agents:

Game Designer: writes a brief for the game.
Game Developer: implements the game by calling Codex MCP.

In [4]:
server_code = """import asyncio
import os

from dotenv import load_dotenv

from agents import Agent, Runner, set_default_openai_api
from agents.mcp import MCPServerStdio

load_dotenv(override=True)
set_default_openai_api()


async def main() -> None:
    async with MCPServerStdio(
        name=\"Codex CLI\",
        params={
            \"command\": "npx",
            "args": ["-y", "codex", "mcp-server-single"],
        },
        client_session_timeout_seconds=360000,
    ) as codex_mcp_server:
        developer_agent = Agent(
            name="Game Developer",
            instructions=(
                "You are an expert in building simple games using basic html + css + javascript with no dependencies. "
                "Save your work in a file called index.html in the current directory. "
                "Always call codex with \\"approval-policy\\": \\"never\\" and \\"sandbox\\": \\"workspace-write\\"."
            ),
            mcp_servers=[codex_mcp_server],
        )

        designer_agent = Agent(
            name="Game Designer",
            instructions=(
                "You are an indie game connoisseur. Come up with an idea for a single page html + css + javascript game that a developer could build in about 50 lines of code. "
                "Format your request as a 3 sentence design brief for a game developer and call the Game Developer coder with your idea."
            ),
            model="gpt-5",
            handoffs=[developer_agent],
        )

        await Runner.run(designer_agent, "Implement a fun new game!")


if __name__ == "__main__":
    asyncio.run(main())
"""
with open("codex_mcp_single.py", "w") as f:
    f.write(server_code)

print("codex_mcp_single.py written")

codex_mcp_single.py written


In [5]:
!python codex_mcp_single.py

Traceback (most recent call last):
  File "D:\Ethans\Python\AI-Automation\codex_mcp_single.py", line 10, in <module>
    set_default_openai_api()
TypeError: set_default_openai_api() missing 1 required positional argument: 'api'


Expand to a multi-agent workflow

Now turn the single-agent setup into an orchestrated, traceable workflow. The system adds:

Project Manager: creates shared requirements, coordinates hand-offs, and enforces guardrails.
Designer, Frontend Developer, Server Developer, and Tester: each with scoped instructions and output folders.

In [9]:
server_code = """
import asyncio
import os

from dotenv import load_dotenv

from agents import (
    Agent,
    ModelSettings,
    Runner,
    WebSearchTool,
    set_default_openai_api,
)
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX
from agents.mcp import MCPServerStdio
from openai.types.shared import Reasoning

load_dotenv(override=True)
set_default_openai_api()


async def main() -> None:
    async with MCPServerStdio(
        name="Codex CLI",
        params={"command": "npx", "args": ["-y", "codex", "mcp"]},
        client_session_timeout_seconds=360000,
    ) as codex_mcp_server:
        designer_agent = Agent(
            name="Designer",
            instructions=(
                f\"""{RECOMMENDED_PROMPT_PREFIX}\"""
                "You are the Designer.\\n"
                "Your only source of truth is AGENT_TASKS.md and REQUIREMENTS.md from the Project Manager.\\n"
                "Do not assume anything that is not written there.\\n\\n"
                "You may use the internet for additional guidance or research."
                "Deliverables (write to /design):\\n"
                "- design_spec.md – a single page describing the UI/UX layout, main screens, and key visual notes as requested in AGENT_TASKS.md.\\n"
                "- wireframe.md – a simple text or ASCII wireframe if specified.\\n\\n"
                "Keep the output short and implementation-friendly.\\n"
                "When complete, handoff to the Project Manager with transfer_to_project_manager."
                "When creating files, call Codex MCP with {\\"approval-policy\\":\\"never\\",\\"sandbox\\":\\"workspace-write\\"}."
            ),
            model="gpt-5",
            tools=[WebSearchTool()],
            mcp_servers=[codex_mcp_server],
        )

        frontend_developer_agent = Agent(
            name="Frontend Developer",
            instructions=(
                f\"""{RECOMMENDED_PROMPT_PREFIX}\"""
                "You are the Frontend Developer.\\n"
                "Read AGENT_TASKS.md and design_spec.md. Implement exactly what is described there.\\n\\n"
                "Deliverables (write to /frontend):\\n"
                "- index.html – main page structure\\n"
                "- styles.css or inline styles if specified\\n"
                "- main.js or game.js if specified\\n\\n"
                "Follow the Designer’s DOM structure and any integration points given by the Project Manager.\\n"
                "Do not add features or branding beyond the provided documents.\\n\\n"
                "When complete, handoff to the Project Manager with transfer_to_project_manager_agent."
                "When creating files, call Codex MCP with {\\"approval-policy\\":\\"never\\",\\"sandbox\\":\\"workspace-write\\"}."
            ),
            model="gpt-5",
            mcp_servers=[codex_mcp_server],
        )

        backend_developer_agent = Agent(
            name="Backend Developer",
            instructions=(
                f\"""{RECOMMENDED_PROMPT_PREFIX}\"""
                "You are the Backend Developer.\\n"
                "Read AGENT_TASKS.md and REQUIREMENTS.md. Implement the backend endpoints described there.\\n\\n"
                "Deliverables (write to /backend):\\n"
                "- package.json – include a start script if requested\\n"
                "- server.js – implement the API endpoints and logic exactly as specified\\n\\n"
                "Keep the code as simple and readable as possible. No external database.\\n\\n"
                "When complete, handoff to the Project Manager with transfer_to_project_manager_agent."
                "When creating files, call Codex MCP with {\\"approval-policy\\":\\"never\\",\\"sandbox\\":\\"workspace-write\\"}."
            ),
            model="gpt-5",
            mcp_servers=[codex_mcp_server],
        )

        tester_agent = Agent(
            name="Tester",
            instructions=(
                f\"""{RECOMMENDED_PROMPT_PREFIX}\"""
                "You are the Tester.\\n
                "Read AGENT_TASKS.md and TEST.md. Verify that the outputs of the other roles meet the acceptance criteria.\\n\\n
                "Deliverables (write to /tests):\\n"
                "- TEST_PLAN.md – bullet list of manual checks or automated steps as requested\\n
                "- test.sh or a simple automated script if specified\\n\\n"
                "Keep it minimal and easy to run.\\n\\n"
                "When complete, handoff to the Project Manager with transfer_to_project_manager."
                "When creating files, call Codex MCP with {\\"approval-policy\\":\\"never\\",\\"sandbox\\":\\"workspace-write\\"}."
            ),
            model="gpt-5",
            mcp_servers=[codex_mcp_server],
        )

        project_manager_agent = Agent(
            name="Project Manager",
            instructions=(
                f\"""{RECOMMENDED_PROMPT_PREFIX}\"""
               \"""
                You are the Project Manager.

                Objective:
                Convert the input task list into three project-root files the team will execute against.

                Deliverables (write in project root):
                - REQUIREMENTS.md: concise summary of product goals, target users, key features, and constraints.
                - TEST.md: tasks with [Owner] tags (Designer, Frontend, Backend, Tester) and clear acceptance criteria.
                - AGENT_TASKS.md: one section per role containing:
                  - Project name
                  - Required deliverables (exact file names and purpose)
                  - Key technical notes and constraints

                Process:
                - Resolve ambiguities with minimal, reasonable assumptions. Be specific so each role can act without guessing.
                - Create files using Codex MCP with {"approval-policy":"never","sandbox":"workspace-write"}.
                - Do not create folders. Only create REQUIREMENTS.md, TEST.md, AGENT_TASKS.md.

                Handoffs (gated by required files):
                1) After the three files above are created, hand off to the Designer with transfer_to_designer_agent and include REQUIREMENTS.md and AGENT_TASKS.md.
                2) Wait for the Designer to produce /design/design_spec.md. Verify that file exists before proceeding.
                3) When design_spec.md exists, hand off in parallel to both:
                   - Frontend Developer with transfer_to_frontend_developer_agent (provide design_spec.md, REQUIREMENTS.md, AGENT_TASKS.md).
                   - Backend Developer with transfer_to_backend_developer_agent (provide REQUIREMENTS.md, AGENT_TASKS.md).
                4) Wait for Frontend to produce /frontend/index.html and Backend to produce /backend/server.js. Verify both files exist.
                5) When both exist, hand off to the Tester with transfer_to_tester_agent and provide all prior artifacts and outputs.
                6) Do not advance to the next handoff until the required files for that step are present. If something is missing, request the owning agent to supply it and re-check.

                PM Responsibilities:
                - Coordinate all roles, track file completion, and enforce the above gating checks.
                - Do NOT respond with status updates. Just handoff to the next agent until the project is complete.
                \"""
            ),
            model="gpt-5",
            model_settings=ModelSettings(
                reasoning=Reasoning(effort="medium"),
            ),
            handoffs=[designer_agent, frontend_developer_agent, backend_developer_agent, tester_agent],
            mcp_servers=[codex_mcp_server],
        )

        designer_agent.handoffs = [project_manager_agent]
        frontend_developer_agent.handoffs = [project_manager_agent]
        backend_developer_agent.handoffs = [project_manager_agent]
        tester_agent.handoffs = [project_manager_agent]

        task_list = \"""
Goal: Build a tiny browser game to showcase a multi-agent workflow.

High-level requirements:
- Single-screen game called "Bug Busters".
- Player clicks a moving bug to earn points.
- Game ends after 20 seconds and shows final score.
- Optional: submit score to a simple backend and display a top-10 leaderboard.

Roles:
- Designer: create a one-page UI/UX spec and basic wireframe.
- Frontend Developer: implement the page and game logic.
- Backend Developer: implement a minimal API (GET /health, GET/POST /scores).
- Tester: write a quick test plan and a simple script to verify core routes.

Constraints:
- No external database—memory storage is fine.
- Keep everything readable for beginners; no frameworks required.
- All outputs should be small files saved in clearly named folders.
\"""

        result = await Runner.run(project_manager_agent, task_list, max_turns=30)
        print(result.final_output)


if __name__ == "__main__":
    asyncio.run(main())
"""
with open("codex_mcp_multiple.py", "w") as f:
    f.write(server_code)

print("codex_mcp_multiple.py written")


codex_mcp_multiple.py written


In [11]:
!python codex_mcp_multiple.py

SyntaxError: Non-UTF-8 code starting with '\x96' in file D:\Ethans\Python\AI-Automation\codex_mcp_multiple.py on line 37, but no encoding declared; see https://peps.python.org/pep-0263/ for details
